# Tool calling: Turning functions into tools

## 1. Introduction

### 1.1 Lab overview

In this lab, you will create a set of tools with the Anthropic SDK to give to an LLM. You will see how the LLM requests tools to be used and also the LLM choosing certain tools when relevant to its task.

This version has four exercises. Complete each `# TODO` in the cells marked Exercise 1 to 4, and run the check cells to test your work. All other cells are complete.

### 1.2 Learning outcome

Apply tool-calling design patterns to assistant workflows.

To achieve this you will give LLMs controlled access to python functions via the Anthropic SDK, manage parameter passing and execution flow, and inspect the tool calls and tool results of multi-step runs.

## 2. Setup: Initialize environment and client

Begin by initializing your environment. You will import several packages now and also later on as you build tools for your LLM.

The next cell installs `anthropic`, sets the endpoint and `MODEL` to `claude-sonnet-5-5`, reads the key from the Colab secret `AZURE_API_KEY`, and creates `client`. Add the secret with the key icon in the sidebar and turn on Notebook access before you run this cell; never paste the key into a cell. Expect no output, or a note about restarting the kernel.

In [ ]:
%pip install -q "anthropic>=1.4.0"
from anthropic import Anthropic
from google.colab import userdata

endpoint = "https://evergreen-ai-dev-resource.services.ai.azure.com/anthropic"
MODEL = "claude-sonnet-5-5"  # the deployment name
api_key = userdata.get("AZURE_API_KEY")  # Colab Secrets (key icon, left sidebar)

client = Anthropic(base_url=endpoint, api_key=api_key)

### 2.1 Getting started with the Anthropic client

The **Anthropic** client that you created above is your interface for generating responses and calling tools. It points to our Foundry endpoint.

Run the cell below to install the other packages that this lab uses later and to import the basics. Expect install messages, or a note about restarting the kernel.

In [ ]:
%pip install -q requests "qrcode[pil]" "Pillow>=10.1"

import json
import os

## 3. Build your first tool

### 3.1 Defining your function

Now that you have set up your environment, it is time to create your first tool. The Anthropic SDK uses the docstring of a function as the tool description, the text that the LLM reads to decide when to call the tool, so the docstring is as important as the code.

**Exercise 1.** Write the function `get_current_time`. It takes no arguments, has a docstring that says what the function does, and returns the current time as a string in the form `HH:MM:SS`. Expect no output.

In [ ]:
from datetime import datetime

# TODO 1: Write get_current_time with a docstring; return the time as HH:MM:SS.
# Hint 1: A string in triple quotes as the first statement of a function is its docstring.
# Hint 2: datetime.now() holds the current time; its strftime method turns it into text from a pattern of codes such as %H.
def get_current_time():
    # YOUR CODE HERE (add the docstring, then return the time)
    return ""

Test out your function to see exactly what this function returns. Requires Exercise 1. Expect the current time as a string in the form `HH:MM:SS`.

<div style="background-color:#ffe4e1; padding:12px; border-radius:6px; color:black;">
  <strong>Note:</strong> The function returns the current time of the machine that runs the notebook, so the value follows that machine's time zone.
</div>

In [ ]:
get_current_time()

Great! Just as expected, the function returns a string that has your current time.

### 3.2 Turning your function into an LLM tool

Now, let's use the Anthropic SDK to pass this tool to an LLM and get a response. To set up your tool, you first set up the `response` from the LLM. Creating a response first requires creating the message structure. The message structure includes the prompt the user asks, as well as a list that represents the conversation history, with each message having a `role` (e.g., "user" or "assistant") and `content`. A system prompt, if you use one, is passed separately as the `system` parameter.

The next cell defines the message structure. Expect no output.

In [ ]:
# Message structure
prompt = "What time is it?"
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

After defining your message structure you can ask the LLM to answer with your tool available. The Anthropic SDK does this with a tool runner, `client.beta.messages.tool_runner`. Let's take a look at the parameters in this call.
* `model`: The model that will be used
* `max_tokens`: The maximum number of tokens the LLM may write in one reply
* `messages`: The list of messages passed to the LLM
* `tools`: The list of tools that the LLM has access to. Each Python function is turned into a tool with `beta_tool`, which reads its docstring and type hints.
* `max_iterations`: This is the step limit: the maximum number of model calls the runner will make. It caps the loop, so the LLM cannot get into an infinite loop of repeated tool calls. Asking for a tool is one call and writing the final answer is another, so use at least 2.

The next cell defines two small helper functions. `run_with_tools` makes this call, runs the tools the LLM asks for, and returns the whole conversation. `final_text` reads the text of the last reply. Expect no output.

In [ ]:
from anthropic import beta_tool

def run_with_tools(messages, functions, max_iterations=10):
    """Let the LLM call the Python functions on its own; return the whole message list."""
    messages = list(messages)                          # work on a copy of the list
    tools = [beta_tool(f) for f in functions]          # schema comes from the docstring and type hints
    runner = client.beta.messages.tool_runner(
        model=MODEL, max_tokens=2048, tools=tools,
        messages=messages, max_iterations=max_iterations)
    for reply in runner:                               # one pass = one model call
        messages.append({"role": "assistant", "content": reply.content})
        tool_results = runner.generate_tool_call_response()  # runs the tools the LLM asked for
        if tool_results:
            messages.append(tool_results)
    return messages

def final_text(messages):
    """Return the text of the last assistant message; thinking and tool blocks are skipped."""
    last = messages[-1]
    if last["role"] != "assistant":
        return ""
    return "".join(b.text for b in last["content"] if b.type == "text")

Run the cell below to call the LLM and see the response. Requires Exercise 1. Expect one sentence that gives the current time.

In [ ]:
conversation = run_with_tools(messages, [get_current_time], max_iterations=5)

# See the LLM response
print(final_text(conversation))

And just like that you've given your LLM access to tools! The tool runner turned your function into a tool that augmented the LLM's knowledge about the world.

### 3.3 Taking a closer look at the response
While the final response's content was just what was expected, there is actually a lot happening behind the scenes in the `conversation`. Let's use a helpful utility function to take a closer look. `show_trace` will extract the steps from the conversation and show you the important parts in an easy to read format.

The next cell defines `show_trace`. Expect no output.

In [ ]:
def show_trace(messages):
    """Print each step of a tool-calling run: text, tool calls, tool results, final answer."""
    def blocks(content):
        if isinstance(content, str):
            return [{"type": "text", "text": content}]
        return [b if isinstance(b, dict) else b.model_dump() for b in content]
    step = 0
    for m in messages:
        for b in blocks(m["content"]):
            if b["type"] == "text" and m["role"] == "user":
                print(f"USER: {b['text']}")
            elif b["type"] == "text":
                print(f"ASSISTANT: {b['text']}")
            elif b["type"] == "tool_use":
                step += 1
                print(f"[{step}] TOOL CALL  {b['name']}({json.dumps(b['input'])})")
            elif b["type"] == "tool_result":
                out = b["content"] if isinstance(b["content"], str) else "".join(c.get("text", "") for c in b["content"])
                print(f"    TOOL RESULT {out[:300]}")
            # "thinking" blocks are skipped on purpose

Run the cell below to print the steps of the conversation from section 3.2. Requires the previous call to have run. Expect the question, one call to `get_current_time` with its result, and the final answer.

In [ ]:
show_trace(conversation)

As you can see, the LLM proposed a tool call to `get_current_time`. The LLM does not run the tool: your code (the app) ran it on your machine and sent the tool result back to the LLM. Finally, the LLM, having the full conversation history, used that information to give the final response. When you passed the function to `run_with_tools`, the tool runner pulled out the `tool_use` block, ran the function locally, and passed the result back to the LLM as a `tool_result`, up to the `max_iterations` limit.

### 3.4 Manually defining tools

You saw that the docstring provided by our tool helped the Anthropic SDK automatically turn your function into a tool for the LLM. This is very convenient. But what is happening behind the scenes to take your function and make it a tool?

Let's first print the tool that `beta_tool` built from `get_current_time`. Requires Exercise 1. Expect a small dictionary with `name`, `description`, and `input_schema`.

In [ ]:
print(json.dumps(beta_tool(get_current_time).to_dict(), indent=2))

Actually, what the tool looks like to the LLM is more complicated. Let's take a look at how the LLM is *actually* given a tool below. Like before, tools are given in a list. But in that list, each tool is a tool definition in a set format that your code sends to the LLM together with the messages. The schema has several important parts:
* `name`: The name of the corresponding function that you defined locally
* `description`: A description that explains what the function does and is used by the LLM to help it decide when to use it
* `input_schema`: A JSON schema for the parameters. If your function has parameters, they would also be described with the parameter name, its type, and a description of what the parameter should be.

**Exercise 2.** Requires Exercise 1. Define `tools` as a list with one dictionary for `get_current_time`, with the three keys above. The function has no parameters. Expect no output.

In [ ]:
# TODO 2: Describe get_current_time as a tool in the Anthropic format.
# Hint 1: The tool is one dictionary in a list; its keys are written exactly as listed above.
# Hint 2: The parameters are described by a JSON schema, which is itself an object; each parameter would be listed under properties.
tools = []  # YOUR CODE HERE (one dictionary for get_current_time)

In this case where you define a schema, you call `client.messages.create` directly and handle the execution yourself. So you will not use the tool runner or `max_iterations`. Let's set this up by defining the response. Requires Exercise 2. Expect no output.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=messages,
    tools=tools,  # <-- Your list of tools with get_current_time
    # When defining tools manually, you must handle calls yourself; there is no step limit
)

Now you can take a look at the response from the LLM. Requires Exercise 2. Expect a `stop_reason` of `tool_use`, followed by the content blocks of the reply.

In [ ]:
print("stop_reason:", response.stop_reason)
for block in response.content:
    data = block.model_dump()
    data.pop("signature", None)  # a long code that is not needed for reading
    print(json.dumps(data, indent=2, default=str))

Notice that the `response` has a `stop_reason` of `tool_use` and a content block of type `tool_use`. This response is the LLM proposing a tool call to `get_current_time`. A tool call is a `tool_use` block with a name, the arguments, and an id. The LLM does not run the tool; your code does. (A `thinking` block can come first; you can ignore it.) You can add some logic to handle this situation. Then pass that back to the model and get the final response.

#### Handle a tool request

**Exercise 3.** Requires Exercise 1. Write `run_tool_request` to handle one tool call, which arrives from the LLM as a `tool_use` block. `functions` is a dictionary that maps a tool name to the Python function. It holds only the tools you registered; the app runs nothing else. Run the function that the block asks for, with the arguments that the LLM chose, and return a `tool_result` block as a dictionary with the keys `type`, `tool_use_id`, and `content`. The `tool_use_id` is the id of the request, and the `content` must be text. Expect no output.

In [ ]:
# TODO 3: Run the requested tool and return the tool_result block.
# Hint 1: The block holds the id of the request (id), the name of the tool (name), and the arguments as a dictionary (input).
# Hint 2: Look up the function by its name, pass the arguments with **, and turn the result into text.
def run_tool_request(block, functions):
    """Runs the function named in a tool_use block and returns the tool_result block."""
    # YOUR CODE HERE
    return {}

Run the cell below to run the function locally with `run_tool_request`, return the result to the LLM, and receive the final response. Requires Exercises 1 to 3. Expect one sentence that gives the current time.

In [ ]:
response2 = None

# Create a condition in case the LLM asked for a tool
if response.stop_reason == "tool_use":
    # Pull out each tool request, run the tool locally, and collect the results
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            tool_results.append(run_tool_request(block, {"get_current_time": get_current_time}))

    if not tool_results or not all(isinstance(r, dict) and r.get("type") == "tool_result" for r in tool_results):
        print("Complete Exercise 3 and run its cell, then run this cell again.")
    else:
        # Send the LLM's turn, including its tool_use blocks, and the results back to the LLM as a user message.
        # A new list is built, so `messages` stays unchanged and this cell can be run again.
        follow_up = messages + [
            {"role": "assistant", "content": response.content},
            {"role": "user", "content": tool_results},
        ]
        response2 = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            messages=follow_up,
            tools=tools,
        )

        print("".join(b.text for b in response2.content if b.type == "text"))
else:
    print("The LLM did not ask for a tool. Complete Exercises 1 and 2, then run the cells from Exercise 2 on again.")

You have now implemented your own manual handling of LLM tool calls. You can choose to have the tools automatically given to your LLM with the tool runner and `max_iterations`, or write the schema and handle the intermediate parts manually. When you handle the calls yourself, your code (the app) is also responsible for checking the arguments, running only the tools you registered, and stopping the loop with a step limit of your own.

### Check your work

This cell defines `check`, `tool_request`, and `text_of`, which test your functions without a model call. Expect no output.

In [ ]:
# Helpers for the check cells; no model call is made.
import inspect
import re
import tempfile
from anthropic import beta_tool
from anthropic.types import ToolUseBlock

def check(n, test, expected):
    """Runs one offline test and prints a message when it fails."""
    try:
        passed, why = bool(test()), ""
    except Exception as e:
        passed, why = False, f" ({type(e).__name__})"
    if not passed:
        print(f"Exercise {n} is incomplete. Expected: {expected}{why}")
    return passed

def tool_request(name, tool_id, **arguments):
    """Builds a tool_use block like the ones the LLM sends."""
    return ToolUseBlock(type="tool_use", id=tool_id, name=name, input=arguments)

def text_of(content):
    """Returns the text of a tool_result content, given as a string or as a list of text blocks."""
    if isinstance(content, str):
        return content
    return "".join(b["text"] for b in content if b["type"] == "text")

This cell checks Exercises 1 to 3 offline. It calls `get_current_time` and `run_tool_request` with made-up data, reads `tools`, and prints each incomplete exercise or a pass message.

In [ ]:
# Check for Exercises 1 to 3; no model call is made.
from datetime import datetime as clock

def seconds_off(text):
    """Returns the distance in seconds between a time HH:MM:SS and the current time of day."""
    def seconds(hms):
        h, m, s = (int(x) for x in hms.split(":"))
        return h * 3600 + m * 60 + s
    diff = abs(seconds(text) - seconds(clock.now().strftime("%H:%M:%S")))
    return min(diff, 86400 - diff)

TIME_FORMAT = r"\d\d:\d\d:\d\d"
TOOL_KEYS = {"name", "description", "input_schema", "type", "cache_control", "strict"}
RESULT_KEYS = {"type", "tool_use_id", "content", "is_error", "cache_control"}
funcs = {"other": lambda **kw: "wrong tool", "add": lambda a, b: a + b, "hello": lambda: "hi there"}

def run_case(name, tool_id, **arguments):
    """Runs run_tool_request on a made-up tool request."""
    return run_tool_request(tool_request(name, tool_id, **arguments), funcs)

shaped = isinstance(tools, list) and len(tools) == 1 and isinstance(tools[0], dict)

def made_result():
    """Returns True when run_tool_request returns a tool_result block for a made-up request."""
    try:
        return run_case("add", "toolu_01A", a=2, b=3).get("type") == "tool_result"
    except Exception:
        return False

done = [
    check(1, lambda: len((inspect.getdoc(get_current_time) or "").split()) >= 3,
          "get_current_time has a docstring that says what the function does."),
    check(1, lambda: re.fullmatch(TIME_FORMAT, get_current_time()) is not None,
          "get_current_time returns the time as a string in the form HH:MM:SS (hours 00 to 23)."),
    check(1, lambda: not re.fullmatch(TIME_FORMAT, get_current_time()) or seconds_off(get_current_time()) < 120,
          "get_current_time returns the current time of the machine."),
    check(2, lambda: shaped,
          "tools is a list with one dictionary for get_current_time."),
    check(2, lambda: not shaped or (tools[0]["name"] == "get_current_time"),
          "name is the name of the function, get_current_time."),
    check(2, lambda: not shaped or (isinstance(tools[0]["description"], str) and len(tools[0]["description"].split()) >= 3),
          "description is a sentence that tells the LLM what the function does."),
    check(2, lambda: not shaped or (isinstance(tools[0]["input_schema"], dict) and tools[0]["input_schema"]["type"] == "object"),
          "input_schema is a JSON schema of type object."),
    check(2, lambda: not shaped or (not tools[0]["input_schema"].get("properties") and not tools[0]["input_schema"].get("required")),
          "input_schema lists no parameters, because get_current_time has none (properties is empty or missing)."),
    check(2, lambda: not shaped or set(tools[0]) <= TOOL_KEYS,
          "the keys are name, description, and input_schema only; the API rejects other keys such as parameters or function."),
    check(3, lambda: run_case("add", "toolu_01A", a=2, b=3).get("type") == "tool_result",
          "the returned dictionary has type tool_result."),
    check(3, lambda: not made_result() or set(run_case("add", "toolu_01A", a=2, b=3)) <= RESULT_KEYS,
          "the dictionary has only the keys type, tool_use_id, and content; the API rejects other keys such as name."),
    check(3, lambda: not made_result() or (run_case("add", "toolu_01A", a=2, b=3)["tool_use_id"] == "toolu_01A"
          and run_case("hello", "toolu_02B")["tool_use_id"] == "toolu_02B"),
          "tool_use_id is the id of the tool_use block."),
    check(3, lambda: not made_result() or text_of(run_case("add", "toolu_01A", a=2, b=3)["content"]) == "5",
          "content is the text of the value that the function returned (the arguments 2 and 3 give the text 5)."),
    check(3, lambda: not made_result() or text_of(run_case("hello", "toolu_02B")["content"]) in ("hi there", '"hi there"'),
          "content is the text that the function returned, here hi there, also when the function takes no arguments."),
]
print("Exercises 1 to 3: all checks passed." if all(done)
      else "Complete the open exercises, then run this cell again.")

## 4. Giving the LLM more tools

Now that you've explored how tools are created and how they are processed locally, let's create a few more tools.

### 4.1 Three new tools

You will define three new tools for your LLM:

- **Weather Tool (`get_weather_from_ip`)**
  Detects the user's location and returns the current, the high, and the low temperature using external API calls to detect your IP address and then send that to a weather API to get the current weather.

- **File Writing Tool (`write_txt_file`)**
  Creates a text file with the specified content in your local environment. The function takes two parameters, `file_path` and `content`.

- **QR Code Generator (`generate_qr_code`)**
  Generates a QR code image from data, with an embedded image. The function takes three parameters: `data`, `filename`, and `image_path`.

The LLM writes the arguments for these parameters when it proposes a tool call.

The weather tool and the QR code tool are given, and you write the file writing tool in Exercise 4. Run the cell below to import `requests` and define the weather tool. Expect no output.

In [ ]:
import requests


def get_weather_from_ip():
    """
    Gets the current, high, and low temperature in Fahrenheit for the user's
    location and returns it to the user.
    """
    # Get location coordinates from the IP address
    lat, lon = requests.get('https://ipinfo.io/json', timeout=10).json()['loc'].split(',')

    # Set parameters for the weather API call
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m",
        "daily": "temperature_2m_max,temperature_2m_min",
        "temperature_unit": "fahrenheit",
        "timezone": "auto"
    }

    # Get weather data
    weather_data = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=10).json()

    # Format and return the simplified string
    return (
        f"Current: {weather_data['current']['temperature_2m']} F, "
        f"High: {weather_data['daily']['temperature_2m_max'][0]} F, "
        f"Low: {weather_data['daily']['temperature_2m_min'][0]} F"
    )

#### Write the file writing tool

**Exercise 4.** Write the tool `write_txt_file(file_path, content)`. It writes `content` into the text file at `file_path`, replaces the file when it exists, and returns the file path as a string. Give each parameter a type hint and the function a docstring, because the Anthropic SDK builds the tool from them. Expect no output.

In [ ]:
# TODO 4: Write write_txt_file so that beta_tool can turn it into a tool.
# Hint 1: The SDK reads the type of each parameter from its type hint (name: type) and the description from the docstring.
# Hint 2: The built-in open function with the mode "w" creates or overwrites a text file; a with block closes the file for you.
def write_txt_file(file_path, content):
    # YOUR CODE HERE (add type hints and a docstring, then write the file)
    return ""

Run the cell below to import the QR code packages and define `generate_qr_code`. Expect no output.

In [ ]:
import qrcode
from qrcode.image.styledpil import StyledPilImage


# Create a QR code
def generate_qr_code(data: str, filename: str, image_path: str):
    """Generate a QR code image given data and an image path.

    Args:
        data: Text or URL to encode
        filename: Name for the output PNG file (without extension)
        image_path: Path to the image to be used in the QR code
    """
    qr = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_H)
    qr.add_data(data)

    img = qr.make_image(image_factory=StyledPilImage, embedded_image_path=image_path)
    output_file = f"{filename}.png"
    img.save(output_file)

    return f"QR code saved as {output_file} containing: {data[:50]}..."

### Check your work

This cell checks Exercise 4 offline. It calls `write_txt_file` on a temporary file, reads the schema that `beta_tool` builds from the function, and prints each incomplete part or a pass message.

In [ ]:
# Check for Exercise 4; no model call is made.
def written(first, second=None):
    """Calls write_txt_file on a temporary file; returns the text read back, the returned value, and the path."""
    with tempfile.TemporaryDirectory() as folder:
        path = os.path.join(folder, "note.txt")
        returned = write_txt_file(file_path=path, content=first)
        if second is not None:
            returned = write_txt_file(file_path=path, content=second)
        if not os.path.exists(path):
            return None, returned, path
        with open(path, encoding="utf-8") as f:
            return f.read(), returned, path

def schema_of(function):
    """Returns the input_schema that beta_tool builds from the function."""
    return beta_tool(function).to_dict()["input_schema"]

done = [
    check(4, lambda: len(beta_tool(write_txt_file).description.split()) >= 3,
          "write_txt_file has a docstring that says what the function does."),
    check(4, lambda: schema_of(write_txt_file)["properties"]["file_path"].get("type") == "string"
          and schema_of(write_txt_file)["properties"]["content"].get("type") == "string",
          "file_path and content have the type hint str, so that the schema gives both the type string."),
    check(4, lambda: {"file_path", "content"} <= set(schema_of(write_txt_file).get("required", [])),
          "file_path and content are the two parameters, and both are required."),
    check(4, lambda: written("line 1\nline 2")[0] == "line 1\nline 2",
          "write_txt_file writes the content into the file at file_path."),
    check(4, lambda: written("first text", "second")[0] == "second",
          "write_txt_file replaces the content of a file that exists."),
    check(4, lambda: (lambda r: isinstance(r[1], str) and r[2] in r[1])(written("x")),
          "write_txt_file returns the file path as a string."),
]
print("Exercise 4: all checks passed." if all(done)
      else "Complete the open exercise, then run this cell again.")

### 4.2 Using your new tools

Now it's time to use your new tools! The conversation will look almost the same, but unlike before, you will pass all the tools to the LLM. The LLM will choose the appropriate tool based on the prompt you send it. Let's start with the `get_weather_from_ip` tool. Requires Exercises 1 and 4, because the list of tools holds your functions. Expect a trace with one call to `get_weather_from_ip` and a final answer that gives the temperatures (this needs internet access).

In [ ]:
prompt = "Can you get the weather for my location?"

conversation = run_with_tools(
    [{"role": "user", "content": (
        prompt
    )}],
    [
        get_current_time,
        get_weather_from_ip,
        write_txt_file,
        generate_qr_code
    ],
    max_iterations=5
)

show_trace(conversation)

You can see that the LLM appropriately chose the correct tool based on the intent of the prompt, even though it had access to other tools.

Now, run the cell below to prompt the LLM to create a note for you. Requires Exercises 1 and 4. Expect a trace with one call to `write_txt_file` and a final answer that confirms the note.

In [ ]:
prompt = "Can you make a txt note for me called reminders.txt that reminds me to call Daniel tomorrow at 7PM?"

conversation = run_with_tools(
    [{"role": "user", "content": (
        prompt
    )}],
    [
        get_current_time,
        get_weather_from_ip,
        write_txt_file,
        generate_qr_code
    ],
    max_iterations=5
)

show_trace(conversation)

And now you have a text file with your reminder in it. You can see that the LLM passed the correct arguments to the tool, the tool ran locally, and then it printed a response that the action was completed. You can even open the text file and read the contents to make sure that it now exists. Requires Exercise 4 and the previous cell. Expect the text of your reminder.

In [ ]:
if os.path.exists("reminders.txt"):
    with open('reminders.txt', 'r') as file:
        contents = file.read()
        print(contents)
else:
    print("reminders.txt does not exist yet. Complete Exercise 4 and run the cell above again.")

Finally, let's use this QR code generation tool to make a neat QR code that takes users to www.example.com.

The QR tool embeds an image in the code, so the next cell first generates a placeholder logo file, `company_logo.jpg`, with Pillow. Expect a small blue image with a white circle and the word LOGO.

In [ ]:
from PIL import Image as PILImage, ImageDraw, ImageFont

# A simple placeholder logo: blue square, white circle, the word LOGO
logo = PILImage.new("RGB", (120, 120), "#0B3D91")
draw = ImageDraw.Draw(logo)
draw.ellipse((10, 10, 110, 110), fill="white")
draw.text((60, 60), "LOGO", fill="#0B3D91", font=ImageFont.load_default(size=26), anchor="mm")
logo.save("company_logo.jpg")
logo

Run the cell below to create the QR code. Requires Exercises 1 and 4. Expect a trace with one call to `generate_qr_code` and a final answer that names the saved file.

In [ ]:
prompt = "Can you make a QR code for me using my company's logo that goes to www.example.com? The logo is located at `company_logo.jpg`. You can call it company_qr_code."

conversation = run_with_tools(
    [{"role": "user", "content": (
        prompt
    )}],
    [
        get_current_time,
        get_weather_from_ip,
        write_txt_file,
        generate_qr_code
    ],
    max_iterations=5
)

show_trace(conversation)

The LLM also successfully passed the correct arguments to the function and then that information was used to run your function and create the QR code. Take a look at the QR by running the cell below and try it out to see if it works by taking you to www.example.com. Expect a QR code image with the placeholder logo in the middle.

In [ ]:
from IPython.display import Image, display

# Display image directly
Image('company_qr_code.png')

### 4.3 Using multiple tools at once

Lastly, it's important to remember that LLMs can use multiple tools at once and go through a sequence of tool calls to accomplish multiple things. Let's use your tools and examine the `conversation` to see what happens. You will use a complex prompt,

> Can you help me create a qr code that goes to www.example.com from the image company_logo.jpg? Also write me a txt note with the current weather please.

This prompt requires a good amount of logic to understand what to call and when. For example, the note needs the current weather, so the LLM has to get the weather first and then pass it to the text note. If the LLM asked to use `write_txt_file` first, it would not have the information from `get_weather_from_ip`. This is a good example of the power of LLMs to parse natural language and use appropriate tools in the right order to accomplish a wide variety of tasks.

<div style="background-color: #ffe4e1; padding: 12px; border-radius: 6px; color: black;">

<h4>What to Notice:</h4>

<ul>
  <li>The LLM <b>automatically chooses</b> which tool to use based on the user request</li>
  <li><b>Parameters are inferred</b> from the user's message (like file names, content, URLs)</li>
  <li>Each tool <b>returns information</b> that the LLM incorporates into its response</li>
  <li><b>No-parameter tools</b> like weather and time are perfect for quick information requests</li>
  <li>The conversation feels <b>natural</b> despite complex operations happening behind the scenes</li>
</ul>

</div>

Run the cell below. Requires Exercises 1 and 4. Expect a trace in which the note is written only after the weather is known, then a final answer.

In [ ]:
prompt = "Can you help me create a qr code that goes to www.example.com from the image company_logo.jpg? Also write me a txt note with the current weather please."

conversation = run_with_tools(
    [{"role": "user", "content": (
        prompt
    )}],
    [
        get_weather_from_ip,
        get_current_time,
        write_txt_file,
        generate_qr_code
    ],
    max_iterations=10
)

show_trace(conversation)

And just like that, you see from the tool sequence that the note was written only after the weather result was known, while the QR code and weather lookups can run together in the first step. After each tool result the LLM decided between another tool call and the final answer, and `max_iterations` was the step limit of that loop. When the LLM asks for several tools in the same step, the trace prints those calls together, followed by their results.

### Model Options

You can experiment with different Claude models when running these tool-calling workflows. This notebook runs on the deployment `claude-sonnet-5-5`; other Claude models can be used when they are deployed in the same resource (change `MODEL`). Each model offers a different balance of capability, cost, and speed:

- **`claude-sonnet-5-5`** - the deployment used in this notebook, strong reasoning and good speed for most tasks
- **Claude Opus models** - the most capable, good for complex multi-step tasks
- **Claude Haiku models** - lighter, faster, and cheaper, good for simpler tasks and quick iterations

Choosing a model depends on your goals:
- Use smaller models for fast, low-cost prototyping
- Switch to stronger models when tasks require better reasoning or multi-step orchestration


### Final Takeaways

- Tool calling lets LLMs go beyond text generation - they can propose calls to functions that your code then runs.
- Clear, well-documented functions (with precise docstrings and typed parameters) help the model know when and how to use each tool. This is how you define tools clearly.
- The Anthropic SDK handles the complexity of translating python functions into tool definitions (`beta_tool`) and orchestrating multi-step workflows (`client.beta.messages.tool_runner`), or you can write the schema and the `tool_use` / `tool_result` handling yourself.
- Every run follows the same lifecycle: your code sends the messages and the tool definitions, the LLM proposes a tool call with arguments, your code runs the tool, the tool result goes back to the LLM, and the LLM gives a final answer or proposes another tool call. The step limit (`max_iterations`) makes sure the loop stops.
- Choosing the right model matters: smaller models are faster and cheaper for simple tasks, while stronger models are better for reasoning-heavy workflows.
- Watching the conversation flow (prompts, tool calls, results, final response) is essential for debugging and improving tool-calling workflows.

With these elements in place, you now have the foundation to design assistants that combine LLM reasoning with external tools to complete more complex tasks.

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

<strong>Congratulations!</strong>

You've completed the lab on **turning functions into tools** with the Anthropic SDK.
Along the way, <strong>you</strong> exposed Python functions as tools, let the LLM choose and call them, and inspected multi-step tool orchestration.

With these skills, <strong>you</strong> can design tool-calling workflows that combine LLM reasoning with real actions - reliable, auditable, and easy to extend.

</div>